# WS-C — Export per-patch cavity attention (Kaggle/GPU)

This notebook produces the **only** artefact WS-C is missing: the per-patch
attention vectors from the trained `SpatialCavityHead`, plus cavity
probabilities from both the spatial and global heads. Everything else
(sextant-label recovery, zone mapping, metrics, controls) runs locally via
`scripts/ws_c_cavity_grounding.py`.

Training is faithful to the repo: same `make_balanced_cavity_split`, same
`SpatialCavityHead`/`ClassifierHead`, same `_train_cav` loop (NAdam, lr 1e-3,
wd 1e-4, CE, best-val-CE checkpoint), averaged over the 5 paper seeds.

**Outputs** (download both, then run the local command in the last cell):
- `attn_spatial.npz` — `image_id [N]`, `attn [N,49]`, `cav_prob [N]` (spatial head)
- `cavprob_global.npz` — `image_id [N]`, `cav_prob [N]` (global CLS head)

**Setup — simplest path:** attach the **whole `dl-project-codebase`** (with the two
`.npz` caches and `tbportals_manifest_paper.csv` inside) as a single Kaggle
**Dataset**. Kaggle auto-unzips it under `/kaggle/input/<your-dataset>/`; the next
cell auto-locates the repo, both caches, and the manifest — no path editing needed.
`/kaggle/input` is read-only, which is fine: this notebook only reads/imports from
there and writes its outputs to `/kaggle/working/ws_c`. GPU is optional (the heads
are tiny). No raw CXR images and no RAD-DINO backbone are needed — the cached
features are sufficient.

In [1]:
# == Config ==
# Easiest setup: attach the WHOLE dl-project-codebase (with the two .npz caches
# and the manifest inside) as a single Kaggle Dataset. Kaggle auto-unzips it under
# /kaggle/input/<your-dataset>/. This cell auto-locates the repo + caches anywhere
# under /kaggle/input (or /kaggle/working); use the manual block if it misses any.
from pathlib import Path

SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]

def _find(name):
    for root in SEARCH_ROOTS:
        if root.exists():
            hits = list(root.rglob(name))
            if hits:
                return sorted(hits, key=lambda p: len(p.parts))[0]  # shallowest match
    return None

GRID_NPZ = _find('features_rad-dino_grid7.npz')
CLS_NPZ  = _find('features_rad-dino_cls.npz')
MANIFEST = _find('tbportals_manifest_paper.csv')
_src     = _find('feature_heads.py')                 # .../src/components/feature_heads.py
REPO     = _src.parents[2] if _src else None          # repo root containing src/
OUT_DIR  = Path('/kaggle/working/ws_c')

# --- manual override (uncomment + edit only if auto-locate misses something) ---
# REPO     = Path('/kaggle/input/<your-dataset>/dl-project-codebase')
# GRID_NPZ = REPO / 'features_rad-dino_grid7.npz'
# CLS_NPZ  = REPO / 'features_rad-dino_cls.npz'
# MANIFEST = REPO / 'notebooks' / 'tbportals_manifest_paper.csv'

HELD_OUTS  = ['Romania', 'Moldova', 'Kazakhstan']
SEEDS      = [0, 1, 2, 3, 4]   # paper uses 5 seeds; attention is averaged over them
EPOCHS     = 150               # matches train_agentic defaults
BATCH_SIZE = 256
LR         = 1e-3
HIDDEN     = 256
VAL_FRAC   = 0.2
OUT_DIR.mkdir(parents=True, exist_ok=True)
for _k in ['REPO', 'GRID_NPZ', 'CLS_NPZ', 'MANIFEST']:
    _v = globals()[_k]
    print(f'{_k:9s}=', _v, '' if (_v and Path(_v).exists()) else '  <-- NOT FOUND, set manually')
assert all(globals()[_k] for _k in ['REPO', 'GRID_NPZ', 'CLS_NPZ', 'MANIFEST']), \
    'one or more inputs not found -- fill in the manual override block above'

REPO     = /kaggle/input/datasets/iahmedhabib/dl-project-codebasess/dl-project-codebase 
GRID_NPZ = /kaggle/input/datasets/iahmedhabib/dl-project-codebasess/dl-project-codebase/features_rad-dino_grid7.npz 
CLS_NPZ  = /kaggle/input/datasets/iahmedhabib/dl-project-codebasess/dl-project-codebase/features_rad-dino_cls.npz 
MANIFEST = /kaggle/input/datasets/iahmedhabib/dl-project-codebasess/dl-project-codebase/notebooks/tbportals_manifest_paper.csv 


In [2]:
# == Imports from the repo (faithful to train_agentic) ==
import sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import torch.nn.functional as F

from src.core.seed import seed_everything
from src.components.feature_heads import SpatialCavityHead, ClassifierHead
from src.data.tbportals import load_manifest, make_country_split
from src.training.train_baseline_paper import make_balanced_cavity_split
from scripts.cache_features import load_features

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)

device: cuda | torch 2.10.0+cu128


In [3]:
# == Load caches + manifest ==
grid_feats, grid_dim = load_features(str(GRID_NPZ))   # image_id -> [49, 768]
cls_feats,  cls_dim  = load_features(str(CLS_NPZ))    # image_id -> [768]
manifest = load_manifest(str(MANIFEST))
print('grid_dim', grid_dim, '| cls_dim', cls_dim, '| manifest', manifest.shape)

grid_dim 768 | cls_dim 768 | manifest (5010, 6)


In [4]:
# == Helpers (mirror train_agentic._gather and _train_cav exactly) ==
def gather(df, feats):
    """Return (X tensor, aligned image_id list). [N,D] for CLS, [N,P,D] for grid."""
    X, ids = [], []
    for iid in df['image_id'].astype(str):
        v = feats.get(iid)
        if v is None:
            continue
        X.append(v); ids.append(iid)
    if not X:
        raise RuntimeError('no features matched this split')
    return torch.tensor(np.stack(X), dtype=torch.float32, device=DEVICE), ids

def train_cav(Xtr, ytr, Xval, yval, in_dim, *, kind, seed):
    """Faithful copy of train_agentic._train_cav (CE, best-val-CE checkpoint)."""
    seed_everything(seed + 100)
    head = (SpatialCavityHead(in_dim, HIDDEN) if kind == 'spatial'
            else ClassifierHead(in_dim, HIDDEN)).to(DEVICE)
    opt = torch.optim.NAdam(head.parameters(), lr=LR, weight_decay=1e-4)
    ytr_l, yval_l = ytr.long(), yval.long()
    N = Xtr.size(0)
    best_state, best_ce = None, float('inf')
    for _ in range(EPOCHS):
        head.train()
        perm = torch.randperm(N, device=DEVICE)
        for i in range(0, N, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            loss = F.cross_entropy(head(Xtr[idx]), ytr_l[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            vce = float(F.cross_entropy(head(Xval), yval_l))
        if vce < best_ce:
            best_ce = vce
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    head.load_state_dict(best_state); head.eval()
    return head, best_ce

def cav_prob(head, X):
    with torch.no_grad():
        return torch.softmax(head(X), dim=1)[:, 1].cpu().numpy()

def attn_of(head, Xgrid):
    """Per-patch attention [N,49] from a SpatialCavityHead."""
    with torch.no_grad():
        _, attn = head(Xgrid, return_attn=True)
    return attn.cpu().numpy()

In [5]:
# == Main loop: per held-out country, 5-seed-averaged attention + probs on TEST images ==
rows_id, rows_attn, rows_ps, rows_pg = [], [], [], []

for country in HELD_OUTS:
    # Test set = every held-out image; identical across seeds (seed only moves train/val).
    _, _, test_df = make_country_split(manifest, country, val_fraction=VAL_FRAC, seed=0)
    Xte_grid, ids_grid = gather(test_df, grid_feats)
    Xte_cls,  ids_cls  = gather(test_df, cls_feats)
    assert ids_grid == ids_cls, 'grid/cls test image_id order mismatch'
    n = len(ids_grid)
    acc_attn = np.zeros((n, Xte_grid.shape[1]), dtype=np.float64)
    acc_ps   = np.zeros(n, dtype=np.float64)
    acc_pg   = np.zeros(n, dtype=np.float64)

    for seed in SEEDS:
        c_tr, c_val, _ = make_balanced_cavity_split(manifest, country, val_fraction=VAL_FRAC, seed=seed)
        # spatial head on the patch grid
        Xct, _ = gather(c_tr, grid_feats); Xcv, _ = gather(c_val, grid_feats)
        yct = torch.tensor(c_tr['cavity'].to_numpy(np.float32), device=DEVICE)
        ycv = torch.tensor(c_val['cavity'].to_numpy(np.float32), device=DEVICE)
        sp_head, sp_ce = train_cav(Xct, yct, Xcv, ycv, grid_dim, kind='spatial', seed=seed)
        acc_attn += attn_of(sp_head, Xte_grid)
        acc_ps   += cav_prob(sp_head, Xte_grid)
        # global head on pooled CLS features
        Xct2, _ = gather(c_tr, cls_feats); Xcv2, _ = gather(c_val, cls_feats)
        gl_head, gl_ce = train_cav(Xct2, yct, Xcv2, ycv, cls_dim, kind='global', seed=seed)
        acc_pg   += cav_prob(gl_head, Xte_cls)
        print(f'  {country} seed {seed}: spatial val-CE {sp_ce:.4f} | global val-CE {gl_ce:.4f}')

    s = len(SEEDS)
    rows_id.extend(ids_grid)
    rows_attn.append(acc_attn / s)
    rows_ps.append(acc_ps / s)
    rows_pg.append(acc_pg / s)
    print(f'{country}: {n} test images, attention averaged over {s} seeds')

[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
  Romania seed 0: spatial val-CE 0.5111 | global val-CE 0.5340
  Romania seed 1: spatial val-CE 0.5266 | global val-CE 0.5293
  Romania seed 2: spatial val-CE 0.5294 | global val-CE 0.5664
  Romania seed 3: spatial val-CE 0.5640 | global val-CE 0.5794
  Romania seed 4: spatial val-CE 0.5358 | global val-CE 0.5446
Romania: 220 test images, attention averaged over 5 seeds
[tbportals] split held_out=Moldova: train=3531 val=890 test=589 (train/val patients 3282/820).
  Moldova seed 0: spatial val-CE 0.5384 | global val-CE 0.5670
  Moldova seed 1: spatial val-CE 0.5355 | global val-CE 0.5460
  Moldova seed 2: spatial val-CE 0.5403 | global val-CE 0.5654
  Moldova seed 3: spatial val-CE 0.5103 | global val-CE 0.5529
  Moldova seed 4: spatial val-CE 0.5788 | global val-CE 0.5888
Moldova: 589 test images, attention averaged over 5 seeds
[tbportals] split held_out=Kazakhstan: train=3690 val=921 test=3

In [6]:
# == Save the two npz files (the WS-C contract) ==
image_id = np.array(rows_id, dtype=object)
attn     = np.concatenate(rows_attn, axis=0)
cav_ps   = np.concatenate(rows_ps, axis=0)
cav_pg   = np.concatenate(rows_pg, axis=0)
# attention rows should already sum to 1 (softmax over 49 patches)
print('attn row-sum range:', float(attn.sum(1).min()), float(attn.sum(1).max()))

np.savez_compressed(OUT_DIR / 'attn_spatial.npz',  image_id=image_id, attn=attn, cav_prob=cav_ps)
np.savez_compressed(OUT_DIR / 'cavprob_global.npz', image_id=image_id, cav_prob=cav_pg)
print('wrote', OUT_DIR / 'attn_spatial.npz', '  shape', attn.shape)
print('wrote', OUT_DIR / 'cavprob_global.npz', '  N', len(image_id))

attn row-sum range: 0.9999999419480445 1.0000000647836713
wrote /kaggle/working/ws_c/attn_spatial.npz   shape (1208, 49)
wrote /kaggle/working/ws_c/cavprob_global.npz   N 1208


## Then, locally

Download `ws_c/attn_spatial.npz` and `ws_c/cavprob_global.npz` from the Kaggle
output, drop them in the repo, and run the local grounding script (no GPU needed):

```bash
python3.11 scripts/ws_c_cavity_grounding.py \
    --attn-npz ws_c/attn_spatial.npz \
    --global-npz ws_c/cavprob_global.npz \
    --out-dir ws_c
```

That writes `ws_c/cavity_grounding_metrics.{json,csv}` with the real zone-AUC,
pointing-game (vs chance), energy concentration, C4 controls (shuffle→0.5,
L/R-flip drops AUC), and the global-vs-spatial calibration (ECE/Brier/reliability).